Пусть:

* $n$ - число выстрелов
* $p_1$ - попадание в критическую зону (мгновенное уничтожение)
* $p_2$ - попадание в топливный бак
* $p_3 = 1 - p_1 - p_2$ - промах
* $k$ - утечка топлива (л/ч) через одну пробоину
* $M$ - критическая потеря топлива

Самолет становится небоеспособным, если:

1. Есть хотя бы одно попадание в критическую зону

   ИЛИ
2. Количество пробоин в баке $r$ такое, что $r \cdot k \ge M$

Критическое количесество попаданий в бак:

$r_{crit} = \left\lceil \frac{M}{k} \right\rceil$

Обозначим:

* $A$ - есть хотя бы одно критическое попадание
* $B$ - число попаданий в бак $\ge r_{crit}$

Тогда искомая вероятность:

$P = 1 - P(\overline{A} \cap \overline{B})$

Вероятность отсутствия критических попаданий:

$P(\overline{A}) = (1 - p_1)^n$

При условии отсутствия критических попаданий, распределение попаданий в бак - биномиальное:

$X \sim Bin\left(n, \frac{p_2}{1-p_1}\right)$

Тогда:

$$ P(\overline{A} \cap \overline{B}) = (1 - p_1)^n \sum_{r=0}^{r_{crit}-1} C_n^r \left(\frac{p_2}{1-p_1}\right)^r \left(\frac{p_3}{1-p_1}\right)^{n-r} $$

$$ P = 1 - (1 - p_1)^n \sum_{r=0}^{r_{crit}-1} C_n^r \left(\frac{p_2}{1-p_1}\right)^r \left(\frac{p_3}{1-p_1}\right)^{n-r} $$

In [2]:
import numpy as np
import math
from pyerf import erfinv

n = 10
p1 = 0.1
p2 = 0.2
p3 = 1 - p1 - p2
k = 5
M = 12
alpha = 0.05

N = 100000
np.random.seed(42)

r_crit = math.ceil(M / k)

count = 0

for _ in range(N):
    shots = np.random.choice([0, 1, 2], size=n, p=[p1, p2, p3])

    if 0 in shots:
        count += 1
    else:
        tank_hits = np.sum(shots == 1)
        if tank_hits >= r_crit:
            count += 1

p_hat = count / N
print(f"Оценка Монте-Карло: {p_hat:.3f}")

analytic_sum = 0

for r in range(r_crit):
    comb = math.comb(n, r)
    term = comb * (p2 / (1 - p1)) ** r * (p3 / (1 - p1)) ** (n - r)
    analytic_sum += term

p_exact = 1 - (1 - p1) ** n * analytic_sum

print(f"Аналитическая вероятность: {p_exact:.3f}")


def quantile(alpha):
    p = (2 - alpha) / 2
    return math.sqrt(2) * erfinv(2 * p - 1)


t = quantile(alpha)
delta = t * (p_exact * (1 - p_exact) / N) ** 0.5

RH = p_exact - delta
RB = p_exact + delta

print(f"Доверительный интервал: [{RH:.3f}, {RB:.3f}]")

Оценка Монте-Карло: 0.786
Аналитическая вероятность: 0.787
Доверительный интервал: [0.785, 0.790]
